# SurfaceTarget-CRC — phase 01 — patch 1 12

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SurfaceTarget-CRC
# Phase 1 — Patch 1.12
#
# DUPLICATE RNA-SEQ QUALITY ASSESSMENT
#
# Objectives:
# 1. Download six duplicate-candidate RNA-seq files
# 2. Verify downloaded file integrity using MD5
# 3. Examine STAR gene-count statistics
# 4. Calculate preliminary RNA-seq quality indicators
# 5. Compare expression profiles within each patient
# 6. Save QC results and provenance
#
# IMPORTANT:
# No final file selection is performed.
# No files are excluded automatically.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import requests
import hashlib
import json
import time
import shutil

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/SurfaceTarget-CRC")

METADATA_DIR = PROJECT_ROOT / "data" / "metadata"

RAW_DIR = (
    PROJECT_ROOT /
    "data/raw/tcga_duplicate_qc"
)

RESULTS_DIR = (
    PROJECT_ROOT /
    "results/tables"
)

RAW_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

AUDIT_PATH = (
    METADATA_DIR /
    "tcga_duplicate_rnaseq_provenance_audit.csv"
)

GDC_DATA_API = "https://api.gdc.cancer.gov/data"

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.12 — DUPLICATE RNA-SEQ QUALITY ASSESSMENT")
print("=" * 65)

# ------------------------------------------------------------
# 2. Load file metadata
# ------------------------------------------------------------

if not AUDIT_PATH.exists():
    raise FileNotFoundError(
        "Duplicate provenance audit not found. "
        "Please complete Patch 1.11."
    )

audit = pd.read_csv(
    AUDIT_PATH,
    dtype=str,
    keep_default_na=False
)

required_columns = [
    "patient_id",
    "file_id",
    "file_size_bytes",
    "md5sum",
    "sample_ids",
    "aliquot_ids",
    "workflow_version"
]

missing = [
    col for col in required_columns
    if col not in audit.columns
]

if missing:
    raise RuntimeError(
        f"Missing metadata columns: {missing}"
    )

if audit["file_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate file identifiers in QC audit."
    )

print("\nMetadata loaded successfully.")

print("Patients:", audit["patient_id"].nunique())

print("Files:", len(audit))

expected_size_mb = (
    pd.to_numeric(audit["file_size_bytes"]).sum()
    / (1024 ** 2)
)

print(f"Expected download: {expected_size_mb:.2f} MB")

# ------------------------------------------------------------
# 3. Check available storage
# ------------------------------------------------------------

disk = shutil.disk_usage("/content")

available_gb = disk.free / (1024 ** 3)

print(f"\nAvailable Colab storage: {available_gb:.2f} GB")

if disk.free < 3 * expected_size_mb * (1024 ** 2):
    raise RuntimeError(
        "Insufficient free storage for safe download."
    )

# ------------------------------------------------------------
# 4. File integrity verification
# ------------------------------------------------------------

def calculate_md5(file_path):

    md5 = hashlib.md5()

    with open(file_path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            md5.update(chunk)

    return md5.hexdigest()


def verify_file(file_path, expected_size, expected_md5):

    if not file_path.exists():
        return False

    actual_size = file_path.stat().st_size

    if actual_size != expected_size:
        return False

    actual_md5 = calculate_md5(file_path)

    return actual_md5.lower() == expected_md5.lower()


# ------------------------------------------------------------
# 5. Download function with retries
# ------------------------------------------------------------

session = requests.Session()

session.headers.update({
    "User-Agent": "SurfaceTarget-CRC-Research/1.0"
})


def download_gdc_file(
    file_id,
    output_path,
    expected_size,
    expected_md5,
    max_attempts=3
):

    if verify_file(
        output_path,
        expected_size,
        expected_md5
    ):

        print("Already downloaded and verified.")

        return

    url = f"{GDC_DATA_API}/{file_id}"

    temporary_path = Path(
        str(output_path) + ".part"
    )

    for attempt in range(1, max_attempts + 1):

        try:

            print(
                f"Download attempt "
                f"{attempt}/{max_attempts}"
            )

            with session.get(
                url,
                stream=True,
                timeout=(30, 180)
            ) as response:

                response.raise_for_status()

                with open(
                    temporary_path,
                    "wb"
                ) as f:

                    for chunk in response.iter_content(
                        chunk_size=1024 * 1024
                    ):

                        if chunk:
                            f.write(chunk)

            # Validate before accepting the file.

            if not verify_file(
                temporary_path,
                expected_size,
                expected_md5
            ):

                raise RuntimeError(
                    "Downloaded file failed "
                    "size or MD5 verification."
                )

            temporary_path.replace(output_path)

            print("Download integrity: VERIFIED")

            return

        except Exception as e:

            print(f"Attempt failed: {e}")

            if temporary_path.exists():
                temporary_path.unlink()

            if attempt == max_attempts:
                raise

            time.sleep(2 ** attempt)


# ------------------------------------------------------------
# 6. Download the six RNA-seq files
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("DOWNLOADING RNA-SEQ FILES")
print("-" * 65)

download_records = []

for index, row in audit.iterrows():

    file_id = row["file_id"]

    patient_id = row["patient_id"]

    expected_size = int(row["file_size_bytes"])

    expected_md5 = row["md5sum"]

    output_path = (
        RAW_DIR /
        f"{file_id}.tsv"
    )

    print("\n" + "-" * 50)

    print(
        f"File {index + 1}/{len(audit)}"
    )

    print("Patient:", patient_id)

    print("File ID:", file_id)

    download_gdc_file(
        file_id,
        output_path,
        expected_size,
        expected_md5
    )

    download_records.append({
        "patient_id": patient_id,
        "file_id": file_id,
        "local_path": str(output_path),
        "expected_md5": expected_md5,
        "file_size_bytes": expected_size,
        "download_verified": True
    })

print("\nAll downloads completed and verified.")

# ------------------------------------------------------------
# 7. RNA-seq count file parsing
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PARSING STAR GENE-COUNT FILES")
print("-" * 65)

STAR_SUMMARY_ROWS = [
    "N_unmapped",
    "N_multimapping",
    "N_noFeature",
    "N_ambiguous"
]


def load_star_counts(file_path):

    # GDC files can contain a commented gene-model
    # header. comment="#" skips such lines.

    df = pd.read_csv(
        file_path,
        sep="\t",
        comment="#",
        low_memory=False
    )

    required = {
        "gene_id",
        "unstranded"
    }

    if not required.issubset(df.columns):

        raise RuntimeError(
            f"Unexpected STAR file format: {file_path}\n"
            f"Columns: {df.columns.tolist()}"
        )

    # Separate STAR count-summary rows.

    summary = df[
        df["gene_id"].isin(STAR_SUMMARY_ROWS)
    ].copy()

    genes = df[
        ~df["gene_id"].isin(STAR_SUMMARY_ROWS)
    ].copy()

    if genes.empty:
        raise RuntimeError(
            "No gene expression rows found."
        )

    if genes["gene_id"].isna().any():
        raise RuntimeError(
            "Missing gene identifiers detected."
        )

    if genes["gene_id"].duplicated().any():
        raise RuntimeError(
            "Duplicate gene identifiers detected."
        )

    genes["unstranded"] = pd.to_numeric(
        genes["unstranded"],
        errors="raise"
    )

    if genes["unstranded"].isna().any():
        raise RuntimeError(
            "Missing gene-count values detected."
        )

    if (genes["unstranded"] < 0).any():
        raise RuntimeError(
            "Negative gene counts detected."
        )

    # Convert summary statistics to numeric.

    summary["unstranded"] = pd.to_numeric(
        summary["unstranded"],
        errors="raise"
    )

    return genes, summary


# ------------------------------------------------------------
# 8. Calculate RNA-seq QC statistics
# ------------------------------------------------------------

qc_records = []

expression_profiles = {}

for _, row in audit.iterrows():

    file_id = row["file_id"]

    patient_id = row["patient_id"]

    file_path = (
        RAW_DIR /
        f"{file_id}.tsv"
    )

    genes, summary = load_star_counts(
        file_path
    )

    # Total counts assigned to genes.

    gene_counts = genes["unstranded"]

    assigned_counts = float(
        gene_counts.sum()
    )

    detected_genes = int(
        (gene_counts > 0).sum()
    )

    total_genes = len(genes)

    # STAR summary statistics.

    star_stats = dict(
        zip(
            summary["gene_id"],
            summary["unstranded"]
        )
    )

    missing_summary = (
        set(STAR_SUMMARY_ROWS)
        - set(star_stats)
    )

    if missing_summary:
        raise RuntimeError(
            f"Missing STAR summary rows: "
            f"{missing_summary}"
        )

    unmapped = float(
        star_stats["N_unmapped"]
    )

    multimapping = float(
        star_stats["N_multimapping"]
    )

    no_feature = float(
        star_stats["N_noFeature"]
    )

    ambiguous = float(
        star_stats["N_ambiguous"]
    )

    # Approximate total counted reads.

    total_counted = (
        assigned_counts
        + unmapped
        + multimapping
        + no_feature
        + ambiguous
    )

    assignment_fraction = (
        assigned_counts / total_counted
        if total_counted > 0
        else np.nan
    )

    # Gene detection percentage.

    detection_fraction = (
        detected_genes / total_genes
        if total_genes > 0
        else np.nan
    )

    # Preserve gene-level counts for comparisons.

    profile = genes.set_index(
        "gene_id"
    )["unstranded"]

    expression_profiles[file_id] = profile

    qc_records.append({

        "patient_id": patient_id,

        "file_id": file_id,

        "sample_ids": row["sample_ids"],

        "aliquot_ids": row["aliquot_ids"],

        "workflow_version": row["workflow_version"],

        "total_genes": total_genes,

        "detected_genes": detected_genes,

        "detection_fraction": detection_fraction,

        "assigned_gene_counts": assigned_counts,

        "unmapped_counts": unmapped,

        "multimapping_counts": multimapping,

        "no_feature_counts": no_feature,

        "ambiguous_counts": ambiguous,

        "total_counted_reads": total_counted,

        "assignment_fraction": assignment_fraction
    })

qc = pd.DataFrame(qc_records)

# ------------------------------------------------------------
# 9. Display quality metrics
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("RNA-SEQ QUALITY METRICS")
print("=" * 65)

qc_display = qc.copy()

qc_display["assigned_counts_millions"] = (
    qc_display["assigned_gene_counts"] / 1e6
)

qc_display["assignment_percent"] = (
    qc_display["assignment_fraction"] * 100
)

qc_display["detected_percent"] = (
    qc_display["detection_fraction"] * 100
)

display_columns = [
    "patient_id",
    "sample_ids",
    "aliquot_ids",
    "assigned_counts_millions",
    "detected_genes",
    "assignment_percent"
]

print("\n")

print(
    qc_display[
        display_columns
    ].round(2).to_string(index=False)
)

# ------------------------------------------------------------
# 10. Compare expression profiles
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("WITHIN-PATIENT EXPRESSION CORRELATION")
print("=" * 65)

correlation_records = []

for patient_id, group in audit.groupby("patient_id"):

    print("\n" + "-" * 60)

    print("Patient:", patient_id)

    file_ids = group["file_id"].tolist()

    # Combine counts by shared gene identifier.

    patient_profiles = pd.concat(
        [
            expression_profiles[fid].rename(fid)
            for fid in file_ids
        ],
        axis=1,
        join="inner"
    )

    if patient_profiles.empty:
        raise RuntimeError(
            f"No shared gene identifiers for {patient_id}"
        )

    # Apply log1p transformation for correlation.
    # This is descriptive QC, not normalization
    # for differential expression.

    log_profiles = np.log1p(
        patient_profiles
    )

    correlation_matrix = (
        log_profiles.corr(method="pearson")
    )

    # Short identifiers for readability.

    short_names = {
        fid: fid[:8]
        for fid in file_ids
    }

    display_matrix = correlation_matrix.rename(
        index=short_names,
        columns=short_names
    )

    print("\nPearson correlation (log1p raw counts):")

    print(
        display_matrix.round(4).to_string()
    )

    # Save each pair once.

    for i in range(len(file_ids)):

        for j in range(i + 1, len(file_ids)):

            fid_a = file_ids[i]

            fid_b = file_ids[j]

            correlation_records.append({

                "patient_id": patient_id,

                "file_a": fid_a,

                "file_b": fid_b,

                "pearson_log1p": (
                    correlation_matrix.loc[
                        fid_a,
                        fid_b
                    ]
                ),

                "shared_genes": len(
                    patient_profiles
                )
            })

correlations = pd.DataFrame(
    correlation_records
)

# ------------------------------------------------------------
# 11. Display QC interpretation reminders
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("QUALITY ASSESSMENT LIMITATIONS")
print("-" * 65)

print(
    "\n1. Higher read depth does not automatically "
    "indicate better biological quality."
)

print(
    "2. Expression correlation is descriptive; "
    "high correlation does not prove identical samples."
)

print(
    "3. Different RNA aliquots may contain "
    "biological or technical variation."
)

print(
    "4. Workflow-version differences require "
    "consideration before combining datasets."
)

print(
    "5. No automatic quality thresholds "
    "or file exclusions have been applied."
)

# ------------------------------------------------------------
# 12. Save QC outputs
# ------------------------------------------------------------

qc_path = (
    RESULTS_DIR /
    "tcga_duplicate_rnaseq_qc_metrics.csv"
)

correlation_path = (
    RESULTS_DIR /
    "tcga_duplicate_rnaseq_correlations.csv"
)

download_manifest_path = (
    METADATA_DIR /
    "tcga_duplicate_qc_download_records.csv"
)

qc.to_csv(
    qc_path,
    index=False
)

correlations.to_csv(
    correlation_path,
    index=False
)

pd.DataFrame(
    download_records
).to_csv(
    download_manifest_path,
    index=False
)

# ------------------------------------------------------------
# 13. Save research provenance
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "1.12",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": "NCI Genomic Data Commons",

    "workflow": "STAR - Counts",

    "files_assessed": len(qc),

    "patients_assessed": int(
        qc["patient_id"].nunique()
    ),

    "count_column": "unstranded",

    "integrity_check": "MD5 checksum and file size",

    "qc_metrics": [
        "assigned_gene_counts",
        "detected_genes",
        "unmapped_counts",
        "multimapping_counts",
        "no_feature_counts",
        "ambiguous_counts",
        "assignment_fraction"
    ],

    "correlation_method": (
        "Pearson correlation of log1p unstranded "
        "gene counts using shared gene identifiers"
    ),

    "file_selection_performed": False,

    "limitations": [
        "QC statistics are preliminary.",
        "No sequencing-level FastQC/Picard review performed.",
        "Read depth alone is not a selection criterion.",
        "Expression similarity does not establish biological equivalence.",
        "Different workflow versions may introduce technical differences.",
        "No files were automatically excluded."
    ]
}

provenance_path = (
    METADATA_DIR /
    "tcga_duplicate_rnaseq_qc_provenance.json"
)

with open(provenance_path, "w") as f:

    json.dump(
        provenance,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 14. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.12 SUMMARY")
print("=" * 65)

print("\nPatients assessed:", qc["patient_id"].nunique())

print("Files assessed:", len(qc))

print(
    "All downloaded files passed integrity verification."
)

print("\nSaved outputs:")

print("1. tcga_duplicate_rnaseq_qc_metrics.csv")

print("2. tcga_duplicate_rnaseq_correlations.csv")

print("3. tcga_duplicate_qc_download_records.csv")

print("4. tcga_duplicate_rnaseq_qc_provenance.json")

print("\nMolecular files stored in:")

print(RAW_DIR)

print("\nNo final file selection performed.")

print("\nPATCH 1.12 COMPLETE")

print("=" * 65)